<a href="https://colab.research.google.com/github/rajiv-ranjan/cds-mini-projects/blob/saif/M6_NB_MiniProject_1_Medical_Q%26A_GPT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Certification Program in Computational Data Science
## A programme by IISc and TalentSprint
### Mini-Project: Medical Q&A using GPT2

## Learning Objectives

At the end of the experiment, you will be able to:

* perform data preprocessing, EDA and feature extraction on the Medical Q&A dataset
* load a pre-trained tokenizer
* finetune a GPT-2 language model for medical question-answering

## Dataset Description

The dataset used in this project is the *Medical Question Answering Dataset* ([MedQuAD](https://github.com/abachaa/MedQuAD/tree/master)). It includes medical question-answer pairs along with additional information, such as the question type, the question *focus*, its UMLS(Unified Medical Language System) details like - Concept Unique Identifier(*CUI*) and Semantic *Type* and *Group*.

To know more about this data's collection, and construction method, refer to this [paper](https://bmcbioinformatics.biomedcentral.com/articles/10.1186/s12859-019-3119-4).

The data is extracted and is in CSV format with below features:

- **Focus**: the question focus
- **CUI**: concept unique identifier
- **SemanticType**
- **SemanticGroup**
- **Question**
- **Answer**

## Part-A: Grading = 10 Points

## Information

Healthcare professionals often have to refer to medical literature and documents while seeking answers to medical queries. Medical databases or search engines are powerful resources of upto date medical knowledge. However, the existing documentation is large and makes it difficult for professionals to retrieve answers quickly in a clinical setting. The problem with search engines and informative retrieval engines is that these systems return a list of documents rather than answers. Instead, healthcare professionals can use question answering systems to retrieve short sentences or paragraphs in response to medical queries. Such systems have the biggest advantage of generating answers and providing hints in a few seconds.

### Problem Statement

Fine-tune gpt2 model on medical-question-answering-dataset for performing response generation for medical queries.

Please refer to ***M6 Assignment-1 Fine-tune GPT2*** to get familiar with how to load pre-trained gpt2 tokenizer and model.

### Import required packages

In [2]:
!pip -q install -U accelerate
!pip -q install -U transformers
!pip -q install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 60.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 85.6 MB/s eta 0:00:00


In [3]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel, TextDataset, DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments

import warnings
warnings.filterwarnings('ignore')

In [4]:
#@title Download the dataset
!wget -q https://cdn.iisc.talentsprint.com/AIandMLOps/MiniProjects/Datasets/MedQuAD.csv
!ls | grep ".csv"

MedQuAD.csv


**Exercise 1: Read the MedQuAD.csv dataset**

**Hint:** pd.read_csv()

In [5]:
df = pd.read_csv("MedQuAD.csv")
df.shape

(16412, 6)

In [6]:
df.head()

,Focus,CUI,SemanticType,SemanticGroup,Question,Answer
0,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,What is (are) Adult Acute Lymphoblastic Leukem...,Key Points - Adult acute lymphoblastic leukemi...
1,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,What are the symptoms of Adult Acute Lymphobla...,"Signs and symptoms of adult ALL include fever,..."
2,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,How to diagnose Adult Acute Lymphoblastic Leuk...,Tests that examine the blood and bone marrow a...
3,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,What is the outlook for Adult Acute Lymphoblas...,Certain factors affect prognosis (chance of re...
4,Adult Acute Lymphoblastic Leukemia,C0751606,T191,Disorders,Who is at risk for Adult Acute Lymphoblastic L...,Previous chemotherapy and exposure to radiatio...


### Pre-processing and EDA

**Exercise 2: Perform below operations on the dataset [0.5 Mark]**

- Handle missing values
- Remove duplicates from data considering `Question` and `Answer` columns

- **Handle missing values**

In [7]:
# YOUR CODE HERE
print(df.isnull().sum())

Focus             14
CUI              565
SemanticType     597
SemanticGroup    565
Question           0
Answer             5
dtype: int64


In [8]:
# Drop missing values
# YOUR CODE HERE
df.dropna(inplace=True)
print(df.shape)
print(df.isnull().sum())

(15810, 6)
Focus            0
CUI              0
SemanticType     0
SemanticGroup    0
Question         0
Answer           0
dtype: int64


- **Remove duplicates from data considering `Question` and `Answer` columns**

In [9]:
# Check duplicates
# YOUR CODE HERE
df.duplicated(subset=['Question', 'Answer']).sum()

np.int64(48)

In [10]:
# Drop duplicates
# YOUR CODE HERE
df.drop_duplicates(subset=['Question', 'Answer'], inplace=True)
print(df.shape)

(15762, 6)


In [11]:
# Check duplicates
# YOUR CODE HERE
df.duplicated(subset=['Question', 'Answer']).sum()

np.int64(0)

**Exercise 3: Display the category name, and the number of records belonging to top 100 categories of `Focus` column [1 Mark]**

In [12]:
# YOUR CODE HERE
# Get top 100 categories in 'Focus' by frequency
_counts = df['Focus'].value_counts().head(100)

focus_df = _counts.reset_index()
focus_df.columns = ['Focus', 'RecordCount']

# Display the result
print(focus_df)

                          Focus  RecordCount
0                 Breast Cancer           53
1               Prostate Cancer           43
2                        Stroke           35
3                   Skin Cancer           34
4           Alzheimer's Disease           30
..                          ...          ...
95      Medullary Sponge Kidney           11
96              IgA Nephropathy           11
97            Alagille Syndrome           11
98  Urinary Incontinence in Men           10
99                  Anal Cancer           10

[100 rows x 2 columns]


In [13]:
# Top 100 Focus categories names
# YOUR CODE HERE
focus_names = df['Focus'].value_counts().head(100).index.tolist()
print(focus_names)

['Breast Cancer', 'Prostate Cancer', 'Stroke', 'Skin Cancer', "Alzheimer's Disease", 'Lung Cancer', 'Colorectal Cancer', 'High Blood Cholesterol', 'Heart Attack', 'Heart Failure', 'High Blood Pressure', "Parkinson's Disease", 'Leukemia', 'Shingles', 'Osteoporosis', 'Hemochromatosis', 'Diabetes', 'Age-related Macular Degeneration', 'Psoriasis', 'Diabetic Retinopathy', 'Gum (Periodontal) Disease', 'Kidney Disease', 'Dry Mouth', 'COPD', 'Cataract', 'Balance Problems', 'Glaucoma', 'Prescription and Illicit Drug Abuse', 'Wilson Disease', 'Gout', 'Medicare and Continuing Care', 'Problems with Taste', 'Short Bowel Syndrome', 'Narcolepsy', 'Endometrial Cancer', 'Neuroblastoma', 'Rheumatoid Arthritis', 'Osteoarthritis', 'Kidney Dysplasia', 'Anxiety Disorders', 'Peripheral Arterial Disease (P.A.D.)', 'Pituitary Tumors', 'Surviving Cancer', 'Problems with Smell', 'Urinary Tract Infections in Children', 'Dry Eye', 'Polycystic Kidney Disease', 'What I need to know about Kidney Failure and How Its T

### Create Training and Validation set

**Exercise 4: Create training and validation set [2 Marks]**

- Consider 4 samples per `Focus` category, for each top 100 categories, from the dataset (It will give 400 samples for training)

- Consider 1 sample per `Focus` category (different from training set), for each top 100 categories, from the dataset (It will give 100 samples for validation)

In [14]:
# YOUR CODE HERE
filtered_df = df[df['Focus'].isin(focus_names)]

filtered_df = filtered_df.sample(frac=1, random_state=42).reset_index(drop=True)

train_samples = (
    filtered_df
    .groupby('Focus', group_keys=False)
    .apply(lambda x: x.iloc[:4])  # First 4 for training
)

remaining_df = filtered_df[~filtered_df.index.isin(train_samples.index)]

val_samples = (
    remaining_df
    .groupby('Focus', group_keys=False)
    .apply(lambda x: x.iloc[:1])  # 1 for validation
)

print("Training samples:", train_samples.shape[0])
print("Validation samples:", val_samples.shape[0])

Training samples: 400
Validation samples: 100


### Pre-process `Question` and `Answer` text

**Exercise 5: Perform below tasks: [1.5 Marks]**

- Combine `Question` and `Answer` for train and validation data as shown below:
    - sequence = *'\<question\>' + question-text + '\<answer\>' + answer-text*

- Join the combined text using '\n' into a single string for training and validation separately

- Save the training and validation strings as separate text files

- **Combine Question and Answer for train and val data**

In [15]:
# YOUR CODE HERE
# Create sequence column in training data
train_samples['sequence'] = (
    '<question>' + train_samples['Question'].astype(str) +
    '<answer>' + train_samples['Answer'].astype(str)
)

# Create sequence column in validation data
val_samples['sequence'] = (
    '<question>' + val_samples['Question'].astype(str) +
    '<answer>' + val_samples['Answer'].astype(str)
)

print(train_samples[['Focus', 'sequence']].head())
print(val_samples[['Focus', 'sequence']].head())

                         Focus  \
21   21-hydroxylase deficiency   
44   21-hydroxylase deficiency   
85   21-hydroxylase deficiency   
204  21-hydroxylase deficiency   
194        Abdominal Adhesions   

                                              sequence  
21   <question>Is 21-hydroxylase deficiency inherit...  
44   <question>What are the treatments for 21-hydro...  
85   <question>How to diagnose 21-hydroxylase defic...  
204  <question>What are the symptoms of 21-hydroxyl...  
194  <question>What are the complications of Abdomi...  
                                            Focus  \
217                     21-hydroxylase deficiency   
798                           Abdominal Adhesions   
1033  Adrenal Insufficiency and Addison's Disease   
494              Age-related Macular Degeneration   
390                             Alagille Syndrome   

                                               sequence  
217   <question>How many people are affected by 21-h...  
798   <question>Wh

- **Join the combined text using '\n' into a single string for training and validation separately**

In [17]:
# YOUR CODE HERE
# Join all sequences into one string for training data
train_text = '\n'.join(train_samples['sequence'].tolist())

# Join all sequences into one string for validation data
val_text = '\n'.join(val_samples['sequence'].tolist())

print(" Train Text Preview:\n", train_text[:500])
print("\n Validation Text Preview:\n", val_text[:500])

🔹 Train Text Preview:
 <question>Is 21-hydroxylase deficiency inherited ?<answer>This condition is inherited in an autosomal recessive pattern, which means both copies of the gene in each cell have mutations. The parents of an individual with an autosomal recessive condition each carry one copy of the mutated gene, but they typically do not show signs and symptoms of the condition.
<question>What are the treatments for 21-hydroxylase deficiency ?<answer>These resources address the diagnosis or management of 21-hydroxy

🔹 Validation Text Preview:
 <question>How many people are affected by 21-hydroxylase deficiency ?<answer>The classic forms of 21-hydroxylase deficiency occur in 1 in 15,000 newborns. The prevalence of the non-classic form of 21-hydroxylase deficiency is estimated to be 1 in 1,000 individuals. The prevalence of both classic and non-classic forms varies among different ethnic populations. 21-hydroxylase deficiency is one of a group of disorders known as congenital adrenal 

- **Save the training and validation strings as text files**

In [18]:
# YOUR CODE HERE
# Optional: Save to text files
with open('train_text.txt', 'w', encoding='utf-8') as f:
    f.write(train_text)

with open('val_text.txt', 'w', encoding='utf-8') as f:
    f.write(val_text)

**Exercise 6: Load pre-trained GPT2Tokenizer [0.5 Mark]**

- Use checkpoint = "gpt2"

In [19]:
# YOUR CODE HERE
checkpoint = "gpt2"
tokenizer = GPT2Tokenizer.from_pretrained(checkpoint)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

**Exercise 7: Tokenize train and validation data and form TextDataset objects [0.5 Mark]**

- Use the loaded pre-trained tokenizer
- Use training and validation data saved in text files

In [20]:
# YOUR CODE HERE
# Tokenize train text
train_dataset = TextDataset(tokenizer=tokenizer, file_path="train_text.txt", block_size=128)

# Tokenize validation text
val_dataset = TextDataset(tokenizer=tokenizer, file_path="val_text.txt", block_size=128)

In [21]:
len(train_dataset), len(val_dataset)

(1103, 238)

In [22]:
# Batch-size
train_dataset[0].shape, val_dataset[0].shape

(torch.Size([128]), torch.Size([128]))

**Exercise 8: Create a DataCollator object [0.5 Mark]**

In [23]:
# YOUR CODE HERE
# Create a Data collator object
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False, return_tensors="pt")

**Exercise 9: Load pre-trained GPT2LMHeadModel [0.5 Mark]**

In [24]:
# YOUR CODE HERE
# Set up the model
model = GPT2LMHeadModel.from_pretrained(checkpoint)

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

**Exercise 10: Fine-tune GPT2 Model [1 Mark]**

- Specify training arguments and create a TrainingArguments object (Use 30 epochs)

- Train a GPT-2 model using the provided training arguments

- Save the resulting trained model and tokenizer to a specified output directory

In [27]:
# Set up the training arguments

# YOUR CODE HERE
model_output_path = "/content/gpt_model"

training_args =  TrainingArguments(
    output_dir = model_output_path,
    overwrite_output_dir = True,
    per_device_train_batch_size = 4, # try with 2
    per_device_eval_batch_size = 4,  #  try with 2
    num_train_epochs = 30,
    save_steps = 1_000,
    save_total_limit = 2,
    logging_dir = './logs',
    )

In [28]:
# Train the model
# YOUR CODE HERE
# Train the model
trainer = Trainer(
    model = model,
    args = training_args,
    data_collator = data_collator,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
)
trainer.train()

# Save the model
# YOUR CODE HERE
trainer.save_model(model_output_path)

# Save the tokenizer
# YOUR CODE HERE
tokenizer.save_pretrained(model_output_path)

Step,Training Loss
500,1.815100
1000,1.513100
1500,1.239300
2000,1.034600
2500,0.874400
3000,0.726500
3500,0.617200
4000,0.529400
4500,0.459300
5000,0.404900


Step,Training Loss
500,1.815100
1000,1.513100
1500,1.239300
2000,1.034600
2500,0.874400
3000,0.726500
3500,0.617200
4000,0.529400
4500,0.459300
5000,0.404900


('/content/gpt_model/tokenizer_config.json',
 '/content/gpt_model/special_tokens_map.json',
 '/content/gpt_model/vocab.json',
 '/content/gpt_model/merges.txt',
 '/content/gpt_model/added_tokens.json')

**Exercise 11: Test Model with user input prompts [1 Mark]**

- Create `generate_response()` function that takes a trained *model*, *tokenizer*, and a *prompt* string as input and generates a response using the GPT-2 model

- Test it with some user input prompts

In [29]:
# YOUR CODE HERE
def generate_response(model, tokenizer, prompt, max_length=100):

    input_ids = tokenizer.encode(prompt, return_tensors="pt")      # 'pt' for returning pytorch tensor

    # Create the attention mask and pad token id
    attention_mask = torch.ones_like(input_ids)
    pad_token_id = tokenizer.eos_token_id

    output = model.generate(
        input_ids,
        max_length=max_length,
        num_return_sequences=1,
        attention_mask=attention_mask,
        pad_token_id=pad_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [30]:
# Load the fine-tuned model and tokenizer

# YOUR CODE HERE
# Load the fine-tuned model and tokenizer
my_model = GPT2LMHeadModel.from_pretrained(model_output_path)
my_tokenizer = GPT2Tokenizer.from_pretrained(model_output_path)

In [31]:
# Response from model

# YOUR CODE HERE
prompt = "What is Oncology?"  # Replace with your desired prompt
response = generate_response(my_model, my_tokenizer, prompt)
print("Generated response:", response)

Generated response: What is Oncology? Cancer has many types. There are many types of cancer, depending on how far you have been treated. There are different types of cancer that start in the liver. There are - secondary cancers (scarring of the liver) - non-kidney carcinoma skin cancer - melanoma skin cancer is a disease in which malignant (cancer) cells form in the tissues of the skin. The skin is the bodys largest organ. It protects against heat, sunlight,


In [32]:
# Testing with given prompt 1

# YOUR CODE HERE
prompt = "What are treatments of cancer?"
response = generate_response(my_model, my_tokenizer, prompt)
print("Generated response:", response)

Generated response: What are treatments of cancer? Treatment for cancer depends on many things, including how far the disease has advanced and what its worth now. Your doctor may be able to recommend treatments that will keep your disease from getting worse.
<question>What are the treatments for Neuroblastoma ?<answer>These resources address the diagnosis or management of neuroblastoma: - Cincinnati Children's Hospital Medical Center - Cleveland Clinic - Genetic Testing Registry: Neuroblastoma Treatment - Gene Review: Neuroblastoma Treatment -


In [35]:
# Testing with given prompt 2

# YOUR CODE HERE
prompt = "What is symtoms of Polio?"
response = generate_response(my_model, my_tokenizer, prompt)
print("Generated response:", response)

Generated response: What is symtoms of Polio? - What is (are) Narcolepsy ?<answer>Narcolepsy is a chronic brain disorder that involves poor control of sleep-wake cycles. People with narcolepsy have episodes of extreme daytime sleepiness and sudden, irresistible bouts of sleep (called "sleep attacks") that can occur at any time, and may last from seconds or minutes. Other signs and symptoms may include cataplexy (a sudden loss of muscle tone


**Exercise 12: Compare the performance of a *GPT2 model* with the *GPT2 model fine-tuned* on MedQuAD data [1 Mark]**

- Load another pre-trained GPT2LMHeadModel and do not fine-tune it

- To generate response using the untuned model, pass it as a parameter to `generate_response()` function

- Test both models (fine-tuned and untuned) with below user input prompts:

    - "What precautions to take for a healthy life?"
    - "What to do after being diagnosed with cancer?"
    - "What to do when feeling sick?"

In [36]:
# Load a pre-trained GPT2 model, do not finetune it with MedQuAD data

# YOUR CODE HERE
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

In [37]:
# Testing with finetuned model: prompt 1

# YOUR CODE HERE
prompt = "What are common remedies for viral fever?"
response = generate_response(my_model, my_tokenizer, prompt)
print("Generated response:", response)

Generated response: What are common remedies for viral fever? Common remedies include the following: - antiviral medications - treating other conditions - avoiding triggers - losing weight - exercising - losing weight - getting older - smoking - taking medications that relax the central nervous system (for example, sedatives and antihistamines) - controlling the production of cortisol such as prednisone, cortisone, or dexamethasone - treating other conditions - losing weight gaining exercising losing weight getting older smoking taking medications that relax the central


In [38]:
# Testing with untuned model: prompt 1

# YOUR CODE HERE
prompt = "What are common remedies for viral fever?"
response = generate_response(model, tokenizer, prompt)
print("Generated response:", response)

Generated response: What are common remedies for viral fever?

The most common remedies for viral fever are:

Antibiotics

Antibiotics are used to treat viral infections. Antibiotics are used to treat viral infections.

Antibiotics are used to treat viral infections. Antibiotics are used to treat viral infections.

Antibiotics are used to treat viral infections. Antibiotics are used to treat viral infections.

Antibiotics are used to treat viral


In [39]:
# Testing with finetuned model: prompt 2

# YOUR CODE HERE
prompt = "How can be asthama prevented?"
response = generate_response(my_model, my_tokenizer, prompt)
print("Generated response:", response)

Generated response: How can be asthama prevented? Yes. Beginning in early childhood, all children should have a comprehensive dilated eye exam at least once a year. The exam checks vision, dilated eye, and upper respiratory tract function. Children who have dilated eye may need to see a doctor within one to two years after the exam to be considered for neuroblastoma. Many children who have neuroblastoma require a comprehensive dilated eye exam at least once a year. If these children need to see


In [40]:
# Testing with untuned model: prompt 2

# YOUR CODE HERE
prompt = "How can be asthama prevented?"
response = generate_response(model, tokenizer, prompt)
print("Generated response:", response)

Generated response: How can be asthama prevented?

Asthama is a form of meditation that is practiced by many people. It is a form of meditation that is practiced by many people. It is a form of meditation that is practiced by many people. It is a form of meditation that is practiced by many people. It is a form of meditation that is practiced by many people. It is a form of meditation that is practiced by many people. It is a form of meditation that is practiced by


In [43]:
# Testing with finetuned model: prompt 3

# YOUR CODE HERE
prompt = "What are medication for flu?"
response = generate_response(my_model, my_tokenizer, prompt)
print("Generated response:", response)

Generated response: What are medication for flu? Men who have medication for flu may need to take medications that relax the muscles of the bladder neck and lower urine pressure to help relieve blockage. These medications include - oxybutynin chloride (Ditropan) - solifenacin (VESIcare) - darifenacin (Enablex) - tolterodine (Detrol) - hyoscyamine (Levsin) - propantheline bromide


In [44]:
# Testing with untuned model: prompt 3

# YOUR CODE HERE
prompt = "What are medication for flu?"
response = generate_response(model, tokenizer, prompt)
print("Generated response:", response)

Generated response: What are medication for flu?

The flu vaccine is a combination of the flu vaccine and the flu vaccine. The flu vaccine is a combination of the flu vaccine and the flu vaccine.

What are the symptoms of flu?

The flu symptoms are usually mild and usually disappear within a few days.

What are the symptoms of flu?

The flu symptoms are usually mild and usually disappear within a few days.

What are the symptoms of flu?

The
